# Predictive Maintenance Experiment

This notebook implements the experiment pipeline as described in "Optimizing Machine Performance and Reliability: A Predictive Maintenance Approach" using the AI4I 2020 dataset.

## 1. Environment Setup
Import necessary libraries and set up the environment.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and Inspect Data
Load the AI4I 2020 dataset and inspect the first few rows.

In [ ]:
# Change this path if needed
df = pd.read_csv('../../instance/baseline_ai4i2020.csv')
df.head()

## 3. Data Preprocessing
1. Drop non-predictive columns.
2. Combine failure modes into a multiclass label and a binary label.
3. Create derived features: Power, PowerWear, TempPerPower.

In [ ]:
# Drop unneeded columns
df_processed = df.drop(columns=['UDI', 'Product ID', 
                                 'TWF', 'PWF', 'RNF', 'HDF', 'OSF'], errors='ignore')

# Define failure priority
priority = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# If the dataset has separate failure mode columns, map using priority order
if set(priority).issubset(df.columns):
    def map_failure(row):
        for mode in priority:
            if row[mode] == 1:
                return mode
        return 'No Failure'

    df_processed['Failure_type'] = df.apply(map_failure, axis=1)
    # Binary label
    df_processed['Machine_Failure'] = df_processed['Failure_type'].apply(lambda x: 0 if x == 'No Failure' else 1)
    # Drop original failure mode columns and the old 'Machine failure' if present
    df_processed = df_processed.drop(columns=priority + ['Machine failure'], errors='ignore')
else:
    # If only a single 'Machine failure' column exists
    df_processed['Machine_Failure'] = df['Machine failure']
    df_processed['Failure_type'] = df['Machine failure'].apply(lambda x: 'No Failure' if x == 0 else 'Failure')

# For multiclass classification, encode failure_type
df_processed['Failure_type_labels'] = df_processed['Failure_type'].astype('category').cat.codes

# Derived features
# ???? Tool wear ??
df_processed['Tool wear'] = df_processed['Tool wear'].replace(0, 1e-6)

# ???? Power?PowerWear?TempPerPower ??
df_processed['Power'] = df_processed['Rotational speed'] * df_processed['Torque']
df_processed['Power'] = df_processed['Power'].replace(0, 1e-6)  # avoid division by zero
df_processed['PowerWear'] = df_processed['Power'] * df_processed['Tool wear']
df_processed['TempPerPower'] = df_processed['Process temperature'] / df_processed['Power']


# Final feature set
df_processed = df_processed[['Air temperature', 'Rotational speed', 'Tool wear',
                             'Torque', 'Process temperature', 'Power', 'PowerWear', 'TempPerPower',
                             'Machine_Failure', 'Failure_type', 'Failure_type_labels']]

df_processed.head()

In [ ]:
X = df_processed.drop(columns=['Machine_Failure', 'Failure_type', 'Failure_type_labels'])
y = df_processed['Machine_Failure'] 
print("=== X.describe() ===")
print(X.describe().T)

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
scaler = StandardScaler()
# ? fit ? transform,???? fit_transform
X_scaled = scaler.fit_transform(X)        # ???? numpy.ndarray
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

df_scaled.to_csv('df_scaled.csv', index=False)

## 5. Train-Test Split
Split into features X and targets y (binary and multiclass). Then stratified 80/20 split.

In [ ]:
# Features for binary classification
X_bin = df_processed[['Air temperature', 'Rotational speed', 'Tool wear',
                      'Torque', 'Process temperature', 'Power', 'PowerWear']]
y_bin = df_processed['Machine_Failure']

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, stratify=y_bin, test_size=0.2, random_state=42)

# Features for multiclass
X_multi = X_bin.copy()
y_multi = df_processed['Failure_type_labels']

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, stratify=y_multi, test_size=0.2, random_state=42)


## 6. Model Training and Evaluation
Train multiple models and evaluate with Accuracy, Precision, Recall, F1-Score.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. ??????:?? KNN?SVM?MLP ???????
models = {
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier())
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=False, random_state=42))
    ]),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', random_state=42),
    'MLP': MLPClassifier(max_iter=10, random_state=42, activation='tanh', learning_rate='constant', learning_rate_init=0.00001)
}

# 2. ?????????????
param_grids = {
    'KNN': {
        'clf__n_neighbors': [3, 5, 7],
        'clf__weights': ['uniform', 'distance']
    },
    'SVM': {
        'clf__C': [0.1, 1, 10],
        'clf__kernel': ['linear', 'rbf']
    },
    'DecisionTree': {
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    },
    'RandomForest': {
        'n_estimators': [50, 100],
        'max_depth': [None, 5]
    },
    'XGBoost': {
        'n_estimators': [50, 100],
        'max_depth': [3, 5],
        'learning_rate': [0.1, 0.01]
    },
    'MLP': {
        'hidden_layer_sizes': [(1)],
        'alpha': [0.001, 0.01]
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

# ?? models ? param_grids ???????????
# models = { 'KNN': KNeighborsClassifier(), ... }
# param_grids = { 'KNN': {...}, ... }

def evaluate_with_progress(X_train, X_test, y_train, y_test, task='binary'):
    results = {}

    # ? tqdm ???????????????
    for name, base_model in tqdm(models.items(), desc="Tuning Models"):
        # ? GridSearchCV ??? verbose=2,???? cross-validation ???
        grid = GridSearchCV(
            estimator=base_model,
            param_grid=param_grids[name],
            scoring='f1_macro',
            cv=7,
            n_jobs=-1,
            verbose=1        # ??????? fold ???
        )

        # ????,?? console ?? verbose ??
        print(f"\n===== Tuning {name} =====")
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        print(f">>> Best params for {name}: {grid.best_params_}\n")

        # ??????????
        y_pred = best_model.predict(X_test)
        y_prob = None
        if hasattr(best_model, 'predict_proba'):
            y_prob = best_model.predict_proba(X_test)[:, 1]

        # ????
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')

        results[name] = {
            'Best_Params': grid.best_params_,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1': f1
        }

    return pd.DataFrame(results).T

# ????(???????)
results_bin = evaluate_with_progress(
    X_train_bin, X_test_bin, y_train_bin, y_test_bin, task='binary'
)
print("\nBinary Classification (with 3-fold CV & Tuning) Results:")
display(results_bin)

# ????????
results_multi = evaluate_with_progress(
    X_train_multi, X_test_multi, y_train_multi, y_test_multi, task='multiclass'
)
print("\nMulticlass Classification (with 3-fold CV & Tuning) Results:")
display(results_multi)

## 7. Visualization
Plot model comparison metrics.

In [ ]:
# ---------------------------------------------------
# 1. ???????? F1 ?????? DataFrame
# ---------------------------------------------------
df_plot = pd.DataFrame({
    'Binary': results_bin['F1'],
    'Multi-class': results_multi['F1']
})

# ---------------------------------------------------
# 2. ???????,???????????
# ---------------------------------------------------
plt.figure(figsize=(10, 6))
bar_width = 0.35
index = list(range(len(df_plot)))

# ?? Binary ??
bars_binary = plt.bar(
    [i - bar_width/2 for i in index],
    df_plot['Binary'],
    width=bar_width,
    label='Binary',
    color='darkgray'
)

# ?? Multi-class ??
bars_multi = plt.bar(
    [i + bar_width/2 for i in index],
    df_plot['Multi-class'],
    width=bar_width,
    label='Multi-class',
    color='black'
)

# ?????????????
for bar in bars_binary:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,  # x ??:????
        height + 0.02,                      # y ??:???????
        f'{height:.4f}',                    # ????:??????
        ha='center',                        # ??????
        va='bottom',                        # ???????
        fontsize=10
    )

for bar in bars_multi:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f'{height:.4f}',
        ha='center',
        va='bottom',
        fontsize=10
    )

# ?????????????
plt.xlabel('Model', fontsize=12)
plt.ylabel('Macro F1 Score', fontsize=12)
plt.title('Comparison of Macro F1 Score: Binary vs Multi-class', fontsize=14)
plt.xticks(index, df_plot.index, fontsize=11)
plt.ylim(0, 1.05)  # y ?? 0 ???? 1,??????????

# ????????
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()